# Detector Optimization via Misclassification Probability
Tsviki Y. Hirsh

March 2nd 2026

Running the full simulation for every candidate parameter set is expensive.
The trick here is to reframe the problem as **classification**, not regression.

## The algorithm in three steps

1. **Label** — split observed simulation results into *good* (below-median cost) and *bad* (above-median).
2. **Classify** — fit `LogisticRegression`. Its `predict_proba` output gives a smooth surface
   $P(\text{bad} \mid \mathbf{x}) \in [0, 1]$ over the parameter space.
   The **distance from the decision boundary** acts as a continuous surrogate signal.
3. **Minimize** — use `mystic` differential evolution to find the point with the lowest
   $P(\text{bad})$, run the true simulation there, add to the dataset, and repeat.

**Parameters optimized:** `gain`, `deadtime`, `blob` for the `image_intensifier_gain` model.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from pathlib import Path
from scipy.spatial import cKDTree

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from mystic.solvers import diffev2

from lumacam import Config, Simulate, Lens, Analysis

## 1. Simulation Pipeline

Three stages wrapped in one function:

| Stage | What it does |
|-------|--------------|
| `Simulate` | Shoots neutrons, records interactions |
| `Lens` | Traces scintillation photons through the optical model using `gain`, `deadtime`, `blob` |
| `Analysis` | Reconstructs events with EMPIR |

Returns `events` (EMPIR-reconstructed events) and `truth` (ground-truth pixel positions
and time-of-flight from the raw simulation).

In [2]:
ARCHIVE = "logistic_opt_run"

bounds      = [(500, 15000), (50, 1000), (0.0, 5.0)]
param_names = ["gain", "deadtime", "blob"]


def run_pipeline(gain, deadtime, blob, num_events=500):
    config = Config.neutrons_tof(energy_min=1.0, energy_max=10.0)
    config.num_events     = num_events
    config.csv_batch_size = num_events
    config.sample_material = "G4Galactic"

    Simulate(archive=ARCHIVE).run(config, verbosity=0)

    Lens(archive=ARCHIVE, verbosity=0).trace_rays(
        detector_model="image_intensifier_gain",
        gain=gain, deadtime=deadtime, blob=blob,
        seed=42, verbosity=0, progress_bar=False,
    )

    Analysis(archive=ARCHIVE).process(
        params="fast_neutrons", export_events=True, verbosity=0,
    )

    csv = sorted((Path(ARCHIVE) / "ExportedEvents").glob("*.csv"))
    events = pd.read_csv(csv[0]) if csv else pd.DataFrame()

    traced_path = Path(ARCHIVE) / "TracedPhotons" / "traced_sim_data_0.csv"
    sim_path    = Path(ARCHIVE) / "SimPhotons"    / "sim_data_0.csv"
    if not traced_path.exists() or not sim_path.exists():
        return events, pd.DataFrame(columns=["neutron_id", "px", "py", "tof_s"])

    traced  = pd.read_csv(traced_path)
    valid   = traced.query("0 <= pixel_x < 256 and 0 <= pixel_y < 256")
    spatial = valid.groupby("neutron_id").agg(
        px=("pixel_x", "mean"), py=("pixel_y", "mean")
    ).reset_index()

    sim_df = pd.read_csv(sim_path)
    timing = sim_df.groupby("neutron_id").agg(
        toa=("toa", "mean"), pulse=("pulse_time_ns", "first")
    ).reset_index()
    timing["tof_s"] = (timing["toa"] - timing["pulse"]) / 1e9

    truth = spatial.merge(timing[["neutron_id", "tof_s"]], on="neutron_id")
    return events, truth

## 2. Reconstruction Cost

For each reconstructed event we find the nearest ground-truth neutron (by pixel position)
and penalise spatial error, temporal error, PSD, and a count mismatch:

$$
\text{cost} =
  \overline{(\hat{x}-x)^2+(\hat{y}-y)^2}
  + \overline{\left(\tfrac{\hat{t}-t}{\sigma_t}\right)^2}
  + \overline{\text{PSD}^2}
  + \left(\tfrac{N_{\text{recon}}-N_{\text{truth}}}{N_{\text{truth}}}\right)^2
$$

Lower cost = better reconstruction quality.

In [3]:
def cost(x):
    gain, deadtime, blob = x
    try:
        events, truth = run_pipeline(gain, deadtime, blob)
    except Exception:
        return 1e6

    if len(events) < 1 or len(truth) < 1:
        return 1e6

    tof_scale = truth["tof_s"].std() or 1e-6
    tree = cKDTree(truth[["px", "py"]].values)
    _, idx = tree.query(events[["x", "y"]].values)
    matched = truth.iloc[idx]

    spatial   = ((events["x"].values - matched["px"].values) ** 2
                + (events["y"].values - matched["py"].values) ** 2).mean()
    temporal  = (((events["tof"].values - matched["tof_s"].values) / tof_scale) ** 2).mean()
    psd       = (events["PSD"].values ** 2).mean()
    count_pen = ((len(events) - len(truth)) / max(len(truth), 1)) ** 2

    return spatial + temporal + psd + count_pen

## 3. Initial Observations

We evaluate the true cost at the **eight corners** of the parameter bounding box ($2^3$ design).
This gives the logistic classifier enough coverage of the space to form a meaningful first
decision boundary before the optimization loop starts.

In [4]:
corners = np.array([[g, d, b]
                    for g in [bounds[0][0], bounds[0][1]]
                    for d in [bounds[1][0], bounds[1][1]]
                    for b in [bounds[2][0], bounds[2][1]]])

y_initial = []
for i, x in enumerate(corners):
    c = cost(x.tolist())
    y_initial.append(c)
    print(f"[{i+1}/{len(corners)}] gain={x[0]:>6.0f}  deadtime={x[1]:>5.0f}  blob={x[2]:.1f}  -> cost={c:.2f}")

y_initial = np.array(y_initial)
X_obs = corners.copy()
y_obs = y_initial.copy()

# Colour-coded summary table
threshold  = np.median(y_initial)
df_initial = pd.DataFrame(corners, columns=param_names)
df_initial["cost"]  = y_initial
df_initial["label"] = np.where(df_initial["cost"] < threshold, "good", "bad")

df_initial.style \
    .background_gradient(subset=["cost"], cmap="RdYlGn_r") \
    .set_caption(f"Initial 2^3 corner observations  (median cost = {threshold:.2f})")

Simulating:   0%|          | 0/500 [00:00<?, ?events/s]

Loading simulation data:   0%|          | 0/1 [00:00<?, ?it/s]

[1/8] gain=   500  deadtime=   50  blob=0.0  -> cost=194.81


Simulating:   0%|          | 0/500 [00:00<?, ?events/s]

Loading simulation data:   0%|          | 0/1 [00:00<?, ?it/s]

...

[7/8] gain= 15000  deadtime= 1000  blob=0.0  -> cost=138.57


Simulating:   0%|          | 0/500 [00:00<?, ?events/s]

Loading simulation data:   0%|          | 0/1 [00:00<?, ?it/s]

[8/8] gain= 15000  deadtime= 1000  blob=5.0  -> cost=191.52


,gain,deadtime,blob,cost,label
0,500.000000,50.000000,0.000000,194.810817,bad
1,500.000000,50.000000,5.000000,159.799052,good
2,500.000000,1000.000000,0.000000,168.566746,bad
3,500.000000,1000.000000,5.000000,190.534995,bad
4,15000.000000,50.000000,0.000000,106.133523,good
5,15000.000000,50.000000,5.000000,160.129436,good
6,15000.000000,1000.000000,0.000000,138.569997,good
7,15000.000000,1000.000000,5.000000,191.521214,bad


## 4. The Logistic Regression Surrogate

### Why logistic regression?

Instead of fitting a regression to the raw cost values, we ask a simpler question:

> Which parameter combinations give good reconstruction?

Logistic regression draws a **decision boundary** that separates good from bad observations.
Its `predict_proba` output is the **sigmoid** of the signed distance from that boundary:

$$P(\text{bad} \mid \mathbf{x})
  = \sigma(\mathbf{w}^\top \mathbf{x} + b)
  = \frac{1}{1+e^{-(\mathbf{w}^\top \mathbf{x}+b)}}$$

This is a smooth surface over the full parameter space — even far from any observed point —
so the optimizer always has a meaningful gradient to follow.
Minimizing $P(\text{bad})$ is equivalent to walking toward the good region.

In [ ]:
# Visualise the sigmoid: distance from boundary -> probability
d = np.linspace(-5, 5, 200)
p = 1 / (1 + np.exp(-d))

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(d, p, color="steelblue", linewidth=2.5)
ax.fill_between(d, 0, p, where=(d < 0),  alpha=0.15, color='green')
ax.fill_between(d, 0, p, where=(d >= 0), alpha=0.15, color='red')
ax.axvline(0, color="gray", linestyle="--", linewidth=1.5, label="decision boundary")
ax.axhline(0.5, color="gray", linestyle=":", linewidth=1)
ax.text(-4.0, 0.12, "good region", fontsize=11, color="green")
ax.text( 1.5, 0.85, "bad region",  fontsize=11, color="red")
ax.set_xlabel("Signed distance from decision boundary")
ax.set_ylabel("P(bad)")
ax.set_title("predict_proba: probability from classifier distance")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def make_lr_surrogate(X, y):
    """
    Fit a logistic classifier on good/bad labels and return a callable
    that outputs P(bad) rescaled to the observed cost range,
    so the optimizer sees cost-like values.
    """
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    labels = (y > np.median(y)).astype(int)   # 0 = good, 1 = bad
    if len(np.unique(labels)) < 2:             # guard: ensure both classes present
        labels[np.argmin(y)] = 0
        labels[np.argmax(y)] = 1

    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_scaled, labels)

    ymin, ymax = y.min(), y.max()

    def surrogate(x):
        p_bad = lr.predict_proba(scaler.transform(np.atleast_2d(x)))[:, 1]
        return float(ymin + (ymax - ymin) * p_bad)

    return surrogate


def plot_surrogate(X, y, title='Logistic surrogate'):
    """2D slice of P(bad) in the gain x deadtime plane (blob fixed at its median)."""
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    labels   = (y > np.median(y)).astype(int)
    if len(np.unique(labels)) < 2:
        labels[np.argmin(y)] = 0; labels[np.argmax(y)] = 1

    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_scaled, labels)

    blob_val = float(np.median(X[:, 2]))
    gain_lin = np.linspace(*bounds[0], 120)
    dt_lin   = np.linspace(*bounds[1], 120)
    GG, DD   = np.meshgrid(gain_lin, dt_lin)
    XY       = np.column_stack([GG.ravel(), DD.ravel(), np.full(GG.size, blob_val)])
    P_bad    = lr.predict_proba(scaler.transform(XY))[:, 1].reshape(GG.shape)

    fig, ax = plt.subplots(figsize=(8, 5))
    cs = ax.contourf(gain_lin, dt_lin, P_bad, levels=25, cmap="RdYlGn_r", vmin=0, vmax=1)
    plt.colorbar(cs, ax=ax, label="P(bad)")
    ax.contour(gain_lin, dt_lin, P_bad, levels=[0.5],
               colors="white", linewidths=2, linestyles="--")

    # Overlay observations near this blob slice
    mask = np.abs(X[:, 2] - blob_val) < 2.5
    if mask.any():
        clrs = ["#2ecc71" if l == 0 else "#e74c3c" for l in labels[mask]]
        ax.scatter(X[mask, 0], X[mask, 1], c=clrs, s=100,
                   edgecolors='k', linewidths=0.8, zorder=5)

    legend_handles = [
        mpatches.Patch(color="#2ecc71", label="good (below-median cost)"),
        mpatches.Patch(color="#e74c3c", label="bad  (above-median cost)"),
        plt.Line2D([0], [0], color='white', lw=2, ls='--', label='decision boundary  P = 0.5'),
    ]
    ax.legend(handles=legend_handles, loc="upper right", framealpha=0.9)
    ax.set_xlabel("Gain")
    ax.set_ylabel("Dead Time [ns]")
    ax.set_title(f"{title}\nblob slice at {blob_val:.1f}  ({len(X)} observations)")
    plt.tight_layout()
    plt.show()

### Surrogate built on initial observations

With only 8 corner points the decision boundary is rough,
but it already captures the gross structure of the cost landscape.

In [ ]:
plot_surrogate(X_obs, y_obs, title="Initial logistic surrogate (8 corner observations)")

## 5. Optimization Loop

Each iteration does three things:

1. **Fit** — build a fresh logistic surrogate on all observations collected so far.
2. **Propose** — minimize $P(\text{bad})$ with `mystic` differential evolution.
   The optimizer only queries the cheap surrogate, never the expensive simulation.
3. **Evaluate** — run the true simulation at the proposed point and add the result to the dataset.

As more data accumulates the classifier becomes sharper, and proposed points
converge toward good parameter combinations.

In [ ]:
n_iter  = 15
history = []

for i in range(n_iter):
    surrogate = make_lr_surrogate(X_obs, y_obs)
    xnew = diffev2(surrogate, bounds, npop=15, bounds=bounds, disp=False)
    y_true = cost(xnew.tolist())

    X_obs = np.vstack([X_obs, xnew])
    y_obs = np.append(y_obs, y_true)
    history.append({**dict(zip(param_names, xnew)), 'cost': y_true})

    marker = "\u2605" if y_true == y_obs.min() else " "
    print(f"{marker} iter {i+1:2d}  "
          f"gain={xnew[0]:7.0f}  deadtime={xnew[1]:5.0f}  blob={xnew[2]:.2f}  "
          f"cost={y_true:.2f}")

## 6. Results

The table shows every parameter combination proposed by the optimizer
together with its true simulation cost. New best values are highlighted in green.

In [ ]:
df_hist = pd.DataFrame(history)
df_hist.index = range(1, len(df_hist) + 1)
df_hist.index.name = "iter"

df_hist.style \
    .highlight_min(subset=["cost"], color="#d4edda") \
    .format({"gain": "{:.0f}", "deadtime": "{:.0f}", "blob": "{:.2f}", "cost": "{:.2f}"}) \
    .set_caption(f"Optimization history  |  best cost = {y_obs.min():.2f}")

In [ ]:
all_costs = np.concatenate([y_initial, df_hist["cost"].values])
cummin    = np.minimum.accumulate(all_costs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# -- Overall convergence --
axes[0].plot(cummin, "o-", color="steelblue", linewidth=2, markersize=5)
axes[0].axvline(len(y_initial) - 0.5, color='gray', linestyle=':',
                linewidth=1.5, label='start of optimization')
axes[0].set_xlabel("Total evaluations")
axes[0].set_ylabel("Best cost found")
axes[0].set_title("Convergence (cumulative best cost)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# -- Cost per optimization iteration --
iter_costs = df_hist["cost"].values
axes[1].plot(range(1, n_iter + 1), iter_costs,
             "s-", color="coral", linewidth=2, markersize=6, alpha=0.8, label="true cost")
axes[1].plot(range(1, n_iter + 1), np.minimum.accumulate(iter_costs),
             "--", color="steelblue", linewidth=2, label="best so far")
axes[1].set_xlabel("Optimization iteration")
axes[1].set_ylabel("True cost")
axes[1].set_title("Cost per Surrogate Iteration")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### How the surrogate evolved

With observations now concentrated near good regions, the classifier has a sharper
view of the cost landscape. Compare the boundary below to the one from the initial 8 points.

In [ ]:
plot_surrogate(X_obs, y_obs, title="Final logistic surrogate (after optimization)")

## 7. Validation at the Best Parameters

One final simulation run at the best parameters found, comparing reconstructed events
to the ground truth spatially and in time-of-flight.

In [ ]:
best_idx = int(np.argmin(y_obs))
xbest    = X_obs[best_idx]

print("Best parameters found:")
print(pd.Series(dict(zip(param_names, xbest))).to_string())
print(f"cost = {y_obs[best_idx]:.4f}")

events_best, truth_best = run_pipeline(*xbest)
print(f"\n{len(truth_best)} neutrons with valid hits -> {len(events_best)} reconstructed events")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(truth_best["px"],  truth_best["py"],
                s=14, alpha=0.7, color='steelblue', label='Truth')
axes[0].scatter(events_best["x"],  events_best["y"],
                s=14, alpha=0.7, color='coral',     label='Reconstructed')
axes[0].set_xlabel("x [px]")
axes[0].set_ylabel("y [px]")
axes[0].set_title("Spatial: Truth vs Reconstructed")
axes[0].set_aspect('equal')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(truth_best["tof_s"] * 1e6,  bins=30, alpha=0.6,
             color='steelblue', edgecolor='white', label='Truth')
axes[1].hist(events_best["tof"]  * 1e6,  bins=30, alpha=0.6,
             color='coral',     edgecolor='white', label='Reconstructed')
axes[1].set_xlabel("Time-of-Flight [\u03bcs]")
axes[1].set_ylabel("Count")
axes[1].set_title("TOF Distribution")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()